# Distance Recalculation

In [ ]:
import pandas as pd
import numpy as np

# Formule mathématique pour calculer la distance entre 2 points GPS en kilomètres
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0 # Rayon de la Terre en km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    # On multiplie par 1.15 pour compenser le fait que les rails ne sont 
    # jamais en ligne droite parfaite (facteur de courbure urbain standard)
    return R * c * 1.15

# 1. Charger votre fichier
file_path = 'delhi-metro-stations.csv'  # Assurez-vous que le chemin est correct
df = pd.read_csv(file_path)

# On s'assure que la colonne Branch existe (au cas où)
if 'Branch' not in df.columns:
    df['Branch'] = 'Main'

# Création d'une nouvelle colonne temporaire pour les calculs
df['Nouvelle_Distance'] = 0.0

# 2. Grouper par Ligne ET par Branche (sort=False permet de garder l'ordre actuel de votre CSV)
for (line, branch), group in df.groupby(['Line', 'Branch'], sort=False):
    distances_cumulees = [0.0] # La première station de cette ligne/branche est toujours à 0 km
    
    # On boucle uniquement sur les stations de CE groupe spécifique
    for i in range(1, len(group)):
        # Station précédente
        lat1 = group.iloc[i-1]['Latitude']
        lon1 = group.iloc[i-1]['Longitude']
        # Station actuelle
        lat2 = group.iloc[i]['Latitude']
        lon2 = group.iloc[i]['Longitude']
        
        # Distance entre ces deux stations
        dist_segment = haversine(lat1, lon1, lat2, lon2)
        
        # Ajout au cumul total
        cumul = distances_cumulees[-1] + dist_segment
        distances_cumulees.append(round(cumul, 2))
        
    # On injecte les distances calculées à leur bonne place dans le grand DataFrame
    df.loc[group.index, 'Nouvelle_Distance'] = distances_cumulees

# 3. Mettre à jour la colonne finale et nettoyer
df['Distance'] = df['Nouvelle_Distance']
df = df.drop(columns=['Nouvelle_Distance'])

# Afficher un petit aperçu pour vérifier
print(df[['Station Name', 'Line', 'Branch', 'Distance']].head(20))

# 4. Sauvegarder le fichier final
df.to_csv(file_path, index=False)
print("🎉 Terminé ! Les distances ont été recalculées et sauvegardées pour TOUTES les lignes et branches.")

        Station Name  Line           Branch  Distance
0           Vaishali  Blue  Vaishali Branch      0.00
1          Kaushambi  Blue  Vaishali Branch      1.82
2        Anand Vihar  Blue  Vaishali Branch      2.78
3         Karkarduma  Blue  Vaishali Branch      3.96
4        Preet Vihar  Blue  Vaishali Branch      5.44
5       Nirman Vihar  Blue  Vaishali Branch      6.57
6        Laxmi Nagar  Blue  Vaishali Branch      7.85
7        Yamuna Bank  Blue  Vaishali Branch      9.28
8   Dwarka Sector 21  Blue             Main      0.00
9    Dwarka Sector 8  Blue             Main      1.97
10   Dwarka Sector 9  Blue             Main      3.10
11  Dwarka Sector 10  Blue             Main      4.33
12  Dwarka Sector 11  Blue             Main      5.47
13  Dwarka Sector 12  Blue             Main      6.70
14  Dwarka Sector 13  Blue             Main      7.73
15  Dwarka Sector 14  Blue             Main      8.79
16            Dwarka  Blue             Main     10.45
17        Dwarka Mor  Blue  